# III-1 全体設計とコストの考え方（ノートブック版）

教科書の III-1 の4つのプログラム（`bench_step.py`・`cost_estimator.py`・`buy_vs_rent.py`・`ledger_summary.py`）を、上から順に実行すれば動く1冊にしたものです。Mac の Jupyter（MPS か CPU）でも、Google Colab（GPU か CPU）でも動きます。執筆時の Mac（Apple シリコン、MPS）で全部で約30秒、CPU だけなら数分です。

| 節 | 試すこと | 教科書の手順 | 使うファイル |
|---|---|---|---|
| 1 | 1ステップの時間を区間つきで測る（U-Net と1次元CNN） | 手順 3・4 | `bench_step.py` |
| 2 | バッチサイズとピークメモリ（CUDA のときだけ直線を当てはめる） | 手順 5 | `bench_step.py` |
| 3 | 費用を区間つきで見積もる（図III-1-1 の仮の値と、1 節の自分の測定） | 手順 6 | `cost_estimator.py` |
| 4 | 買う場合と借りる場合の損益分岐（図III-1-2） | 手順 7 | `buy_vs_rent.py` |
| 5 | 費用の台帳を集計する（見積もりと実際の比の区間） | 手順 8・9 | `ledger_summary.py`・`cost_ledger.csv`・`service_survey.csv` |
| 6 | 原理の式の数値を確かめる（VRAM の見積もり、学習時間、止め忘れ） | III-1-3 | ― |

- 次のセルたちが、III-1-6 のファイルをこのノートブックの隣に `%%writefile` で書き出します。**中身は教科書と1文字も変えていません**。
- ターミナルの `uv run python bench_step.py …` の代わりに、`run("bench_step.py …")` で同じ引数を渡して別のプロセスで実行します。
- **1・2 節は小さい設定が既定**です（U-Net の画像の一辺 128、`--steps 30 --warmup 5`。GPU も MPS もない CPU では、さらに一辺 64、1次元CNN のバッチ 32）。教科書の設定（`--model unet --batch 8 --size 256`、`--steps 50`、発熱を見るなら `--steps 600`）は、セルのコメントにあります。時間の値は機械で大きく変わるので、`ok()` は、設定や機械に依らず成り立つこと（JSON の形、区間の幅の計算、費用の式）だけを確かめます。
- 3・4 節は、本文の図III-1-1・図III-1-2 の**仮の値**（本番 100,000 ステップ、60 円/時、予算 500 円、Mac 1.5 秒・GPU 0.18 秒、PC 315,000 円など）で計算し、図の説明の数値（90%区間 160〜535 円・予算を超える確率 約7%、330〜340 円、約 6,409 時間・約 5,151 時間）と照らし合わせます。料金は仮の値です。実際の料金は各サービスの公式サイトで要確認です。
- 5 節の台帳は、教科書の見本の1行に、ノートブック版で**架空の3行**を足して、「見積もりと実際の比の区間」（3件以上で表示）が出るところまで試します。自分の台帳では、架空の行を消してください。
- **入れていないもの**：GitHub にリポジトリを作ってプッシュすること（手順 2。`.gitignore` と `.env.example` は下に読むためだけに載せてあります）、保護者と予算を決めること（手順 1）、サービスの公式サイトでの調査（手順 8）、クラウドの GPU を借りること。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には torch・numpy・pandas・scipy が最初から入っている。版は公式で要確認）
import importlib, subprocess, sys
for mod, pkg in [("torch", "torch"), ("numpy", "numpy"), ("pandas", "pandas"), ("scipy", "scipy")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import torch, numpy, pandas, scipy, platform
print("OK Python", platform.python_version(), " torch", torch.__version__, " numpy", numpy.__version__,
      " pandas", pandas.__version__, " scipy", scipy.__version__)
print("CUDA:", torch.cuda.is_available(), " MPS:", torch.backends.mps.is_available())
print("（教科書は torch 2.14.1 で確かめた。Colab の torch の版は違うことがある）")

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

**リポジトリの準備（Mac のターミナル。読むだけ）**

```bash
mkdir -p ~/home-cloud && cd ~/home-cloud
git init -b main                 # ブランチ名を main にする（Mac の git の既定は master のことがある）
uv init --python 3.12
uv add torch numpy pandas scipy statsmodels matplotlib
# .gitignore（第II部の内容に、第III部用を追加）
cat >> .gitignore <<'EOF'
.env
*.env
!.env.example
runs/
data/
*.pt
*.ckpt
wandb/
mlruns/
__pycache__/
.venv/
.ipynb_checkpoints/
EOF
cp /dev/null .env.example     # 空のファイルを作る。下の .env.example の内容を貼って保存してから、次の行に進む
git add -A && git commit -m "init home-cloud"
# GitHub で private リポジトリ home-cloud を作ってから
git remote add origin git@github.com:YOURNAME/home-cloud.git
git push -u origin main
```

**`.env.example`（見本。本物の `.env` は Git に入れない）**

```ini
# このファイルをコピーして .env を作り、値を入れる（.env は .gitignore で除外済み）
HF_TOKEN=
WANDB_API_KEY=
VAST_API_KEY=
```

次の6つのセルで、III-1-6 のプログラムと CSV の見本を書き出します。

In [ ]:
%%writefile bench_step.py
#!/usr/bin/env python3
"""III-1：学習1ステップの所要時間とピークメモリを測る（Mac の MPS／CPU、クラウドの CUDA で共通）
使い方:
  uv run python bench_step.py --model unet --batch 8 --size 256 --out bench_mac_unet.json
  uv run python bench_step.py --model audio --batch 128 --size 4096 --out bench_mac_audio.json
  python bench_step.py --model unet --batch 8 --size 256 --out bench_gpu_unet.json   # クラウドで
"""
import argparse
import contextlib
import json
import math
import platform
import statistics
import time

import torch
import torch.nn as nn
import torch.nn.functional as F


def pick_device(name="auto"):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def sync(dev):
    """GPU は非同期に動くので、時間を測る前後で計算の完了を待つ"""
    if dev.type == "cuda":
        torch.cuda.synchronize()
    elif dev.type == "mps":
        torch.mps.synchronize()


def device_name(dev):
    if dev.type == "cuda":
        return torch.cuda.get_device_name(0)
    if dev.type == "mps":
        return "Apple MPS (" + platform.machine() + ")"
    return platform.processor() or platform.machine()


def peak_mem_mb(dev):
    try:
        if dev.type == "cuda":
            return torch.cuda.max_memory_allocated() / 2**20
        if dev.type == "mps":
            return torch.mps.driver_allocated_memory() / 2**20   # キャッシュも含む目安
    except Exception:
        pass
    return None


class Block(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.net(x)


class TinyUNet(nn.Module):
    """3段の小さな U-Net（入力画像の一辺は 8 の倍数）"""
    def __init__(self, cin=1, ncls=2, w=32):
        super().__init__()
        self.e1, self.e2, self.e3 = Block(cin, w), Block(w, 2 * w), Block(2 * w, 4 * w)
        self.mid = Block(4 * w, 8 * w)
        self.u3, self.d3 = nn.ConvTranspose2d(8 * w, 4 * w, 2, stride=2), Block(8 * w, 4 * w)
        self.u2, self.d2 = nn.ConvTranspose2d(4 * w, 2 * w, 2, stride=2), Block(4 * w, 2 * w)
        self.u1, self.d1 = nn.ConvTranspose2d(2 * w, w, 2, stride=2), Block(2 * w, w)
        self.out = nn.Conv2d(w, ncls, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(F.max_pool2d(e1, 2))
        e3 = self.e3(F.max_pool2d(e2, 2))
        m = self.mid(F.max_pool2d(e3, 2))
        d3 = self.d3(torch.cat([self.u3(m), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)


class AudioCNN(nn.Module):
    """波形（1ch）を直接入れる小さな1次元CNN（2クラス分類）"""
    def __init__(self, w=32, ncls=2):
        super().__init__()
        layers, cin = [], 1
        for cout in (w, w, 2 * w, 2 * w, 4 * w, 4 * w):
            layers += [nn.Conv1d(cin, cout, 9, stride=2, padding=4),
                       nn.BatchNorm1d(cout), nn.ReLU(inplace=True)]
            cin = cout
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(cin, ncls)

    def forward(self, x):
        return self.head(self.features(x).mean(dim=-1))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", choices=["unet", "audio"], default="unet")
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--size", type=int, default=256, help="unet: 画像の一辺、audio: 波形の長さ")
    ap.add_argument("--width", type=int, default=32)
    ap.add_argument("--warmup", type=int, default=10)
    ap.add_argument("--steps", type=int, default=50)
    ap.add_argument("--amp", action="store_true", help="CUDA で混合精度を使う")
    ap.add_argument("--device", default="auto")
    ap.add_argument("--out", default="bench.json")
    a = ap.parse_args()
    if a.steps < 30:
        print("注意：--steps が 30 未満だと、区間の計算（正規近似）が粗くなります")

    dev = pick_device(a.device)
    torch.manual_seed(0)
    if dev.type == "cuda":
        torch.backends.cudnn.benchmark = True
        torch.cuda.reset_peak_memory_stats()
    if a.model == "unet":
        model = TinyUNet(w=a.width).to(dev)
        x = torch.randn(a.batch, 1, a.size, a.size, device=dev)
        y = torch.randint(0, 2, (a.batch, a.size, a.size), device=dev)
    else:
        model = AudioCNN(w=a.width).to(dev)
        x = torch.randn(a.batch, 1, a.size, device=dev)
        y = torch.randint(0, 2, (a.batch,), device=dev)
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    use_amp = a.amp and dev.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    times = []
    for i in range(a.warmup + a.steps):
        sync(dev)
        t0 = time.perf_counter()
        opt.zero_grad(set_to_none=True)
        ctx = torch.autocast("cuda", dtype=torch.float16) if use_amp else contextlib.nullcontext()
        with ctx:
            loss = F.cross_entropy(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sync(dev)
        if i >= a.warmup:
            times.append(time.perf_counter() - t0)

    n = len(times)
    m = statistics.mean(times)
    sd = statistics.stdev(times)
    half = 1.96 * sd / math.sqrt(n)
    first, second = times[: n // 2], times[n // 2:]
    res = {
        "device": dev.type, "device_name": device_name(dev), "torch": torch.__version__,
        "model": a.model, "batch": a.batch, "size": a.size, "width": a.width, "amp": use_amp,
        "params": n_params, "warmup": a.warmup, "steps": n,
        "step_mean_s": m, "step_sd_s": sd, "step_ci95_s": [m - half, m + half],
        "step_mean_first_half_s": statistics.mean(first),
        "step_mean_second_half_s": statistics.mean(second),
        "samples_per_s": a.batch / m, "peak_mem_mb": peak_mem_mb(dev),
        "time": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
    }
    with open(a.out, "w", encoding="utf-8") as f:
        json.dump(res, f, ensure_ascii=False, indent=2)
    print(f"{res['device_name']}  {a.model} batch={a.batch} size={a.size}  パラメータ {n_params/1e6:.2f}M")
    print(f"1ステップ = {m*1e3:.1f} ms（95%区間 {(m-half)*1e3:.1f}〜{(m+half)*1e3:.1f} ms、n={n}）")
    print(f"前半 {res['step_mean_first_half_s']*1e3:.1f} ms ／ 後半 {res['step_mean_second_half_s']*1e3:.1f} ms"
          "（後半が遅ければ発熱による速度低下を疑う）")
    pm = res["peak_mem_mb"]
    mem = f"ピークメモリ ≈ {pm:.0f} MB" if pm is not None else "ピークメモリ：（CPU では測らない）"
    print(f"処理量 = {res['samples_per_s']:.1f} サンプル/秒、{mem}")
    print(f"保存: {a.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile cost_estimator.py
#!/usr/bin/env python3
"""III-1：ベンチマーク結果から、クラウドGPUでの所要時間と費用を「区間つき」で見積もる
料金は自分で公式サイトを見て入力する（この教科書は料金を断定しない）。
使い方の例:
  # GPU で 10 分だけ測った結果がある場合
  uv run python cost_estimator.py --bench bench_gpu_unet.json --steps 30000 --price-jpy-h 60 --budget-jpy 1000
  # Mac の結果しかない場合（GPU での速度倍率を「5〜20倍のどこか」と幅で与える）
  uv run python cost_estimator.py --bench bench_mac_unet.json --speedup 5 20 --steps 30000 --price-jpy-h 60 --budget-jpy 1000
"""
import argparse
import json
import math

import numpy as np


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--bench", required=True, help="bench_step.py の出力 JSON")
    ap.add_argument("--steps", type=int, required=True, help="本番の総ステップ数（エポック数 × 1エポックのステップ数）")
    ap.add_argument("--speedup", type=float, nargs=2, default=None, metavar=("LO", "HI"),
                    help="Mac の結果から推定するとき、GPU での速度倍率が 90%% の確からしさで入る範囲")
    ap.add_argument("--price-jpy-h", type=float, required=True, help="GPU の時間単価 [円/時]（公式サイトで確認した値）")
    ap.add_argument("--setup-min", type=float, default=20, help="環境構築・データ転送などの時間 [分]")
    ap.add_argument("--idle-min", type=float, default=15, help="監視・後片付けなど、計算していない時間 [分]")
    ap.add_argument("--disk-gb", type=float, default=0, help="借りるストレージの容量 [GB]")
    ap.add_argument("--disk-jpy-gb-month", type=float, default=0, help="ストレージ単価 [円/GB/月]")
    ap.add_argument("--keep-days", type=float, default=0, help="ストレージを残しておく日数")
    ap.add_argument("--egress-gb", type=float, default=0, help="転送量 [GB]")
    ap.add_argument("--egress-jpy-gb", type=float, default=0, help="転送単価 [円/GB]")
    ap.add_argument("--budget-jpy", type=float, default=None, help="今回の予算 [円]")
    ap.add_argument("--n-sim", type=int, default=100_000)
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()

    with open(a.bench, encoding="utf-8") as f:
        b = json.load(f)
    rng = np.random.default_rng(a.seed)
    # 平均ステップ時間の不確かさ（中心極限定理による正規近似）
    se = b["step_sd_s"] / math.sqrt(b["steps"])
    step = np.clip(rng.normal(b["step_mean_s"], se, a.n_sim), 1e-9, None)
    if a.speedup:
        lo, hi = a.speedup
        mu = (math.log(lo) + math.log(hi)) / 2
        sigma = (math.log(hi) - math.log(lo)) / (2 * 1.645)   # 90% 範囲 → 対数正規分布の σ
        step = step / np.exp(rng.normal(mu, sigma, a.n_sim))
    run_h = step * a.steps / 3600
    gpu_h = run_h + (a.setup_min + a.idle_min) / 60
    cost = (a.price_jpy_h * gpu_h
            + a.disk_gb * a.disk_jpy_gb_month * a.keep_days / 30
            + a.egress_gb * a.egress_jpy_gb)

    p5, p50, p95 = np.percentile(run_h, [5, 50, 95])
    c5, c50, c95 = np.percentile(cost, [5, 50, 95])
    print(f"ベンチマーク: {b['device_name']}（{b['model']}, batch={b['batch']}）")
    print(f"計算時間   : 中央値 {p50:.2f} 時間（90%区間 {p5:.2f}〜{p95:.2f} 時間）")
    print(f"費用       : 中央値 {c50:,.0f} 円（90%区間 {c5:,.0f}〜{c95:,.0f} 円）")
    print(f"  内訳の目安：GPU {a.price_jpy_h*np.median(gpu_h):,.0f} 円、"
          f"ストレージ {a.disk_gb*a.disk_jpy_gb_month*a.keep_days/30:,.0f} 円、転送 {a.egress_gb*a.egress_jpy_gb:,.0f} 円")
    if a.budget_jpy:
        print(f"予算 {a.budget_jpy:,.0f} 円を超える確率 ≈ {(cost > a.budget_jpy).mean()*100:.1f} %")
    print(f"参考：止め忘れ12時間の費用 = {a.price_jpy_h*12:,.0f} 円")
    print("※ 見積もりはステップ時間の測定誤差と速度倍率の幅だけを考えている。"
          "データ読み込みの遅れや中断は含まないので、実際との比を台帳で確かめる。")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile buy_vs_rent.py
#!/usr/bin/env python3
"""III-1：PCを買う場合とクラウドを借りる場合の損益分岐
既定値はすべて「仮の値」。自分で調べた値（電気料金の契約単価など）に置き換えて使う。
使い方: uv run python buy_vs_rent.py --pc-jpy 315000 --cloud-jpy-h 30 60 120
"""
import argparse


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--pc-jpy", type=float, default=315000, help="PCの購入価格 [円]")
    ap.add_argument("--resale-jpy", type=float, default=0, help="数年後に売るときの価格 [円]（控えめに0）")
    ap.add_argument("--watt", type=float, default=350, help="学習中のPC全体の消費電力 [W]（仮の値）")
    ap.add_argument("--elec-jpy-kwh", type=float, default=31, help="電気料金 [円/kWh]（契約で要確認）")
    ap.add_argument("--cloud-jpy-h", type=float, nargs="+", default=[30, 60, 120], help="クラウドの時間単価 [円/時]（仮の値）")
    ap.add_argument("--cloud-overhead", type=float, default=1.2, help="環境構築などで実際に借りる時間の倍率")
    ap.add_argument("--hours-per-week", type=float, nargs="+", default=[2, 5, 10])
    a = ap.parse_args()

    p_e = a.watt / 1000 * a.elec_jpy_kwh
    print(f"自宅PCの電気代 ≈ {p_e:.1f} 円/時（{a.watt:.0f} W × {a.elec_jpy_kwh} 円/kWh）")
    print(f"PCの実質負担 = {a.pc_jpy - a.resale_jpy:,.0f} 円\n")
    header = "クラウド単価 | 損益分岐 [時間] | " + " | ".join(f"週{h:g}時間なら" for h in a.hours_per_week)
    print(header)
    print("-" * len(header))
    for p in a.cloud_jpy_h:
        eff = p * a.cloud_overhead - p_e
        if eff <= 0:
            print(f"{p:6.0f} 円/時 | クラウドのほうが常に安い")
            continue
        h_star = (a.pc_jpy - a.resale_jpy) / eff
        years = [h_star / (h * 52) for h in a.hours_per_week]
        print(f"{p:6.0f} 円/時 | {h_star:12,.0f}   | " + " | ".join(f"{y:7.1f} 年" for y in years))
    print("\n※ 式に入らない要素：PCの他の用途、GPUの値下がりと世代交代、研究室配属後は研究室の計算機が使えること。")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile ledger_summary.py
#!/usr/bin/env python3
"""III-1：費用の台帳を集計する（月ごとの合計と予算使用率、見積もりと実際の比の区間）
使い方: uv run python ledger_summary.py cost_ledger.csv 3000
"""
import sys

import numpy as np
import pandas as pd
from scipy import stats

path = sys.argv[1] if len(sys.argv) > 1 else "cost_ledger.csv"
budget = float(sys.argv[2]) if len(sys.argv) > 2 else 3000.0
df = pd.read_csv(path, parse_dates=["date"])
df["total_jpy"] = df["actual_jpy"].fillna(0) + df["storage_jpy"].fillna(0)

print(f"月の予算: {budget:,.0f} 円")
for month, v in df.groupby(df["date"].dt.to_period("M"))["total_jpy"].sum().items():
    rate = v / budget * 100
    mark = "  ⚠ 80%を超えました" if rate >= 80 else ""
    print(f"  {month}: {v:,.0f} 円（予算の {rate:.0f}%）{mark}")

# 見積もりと実際の比（有料で、両方が記録されている行だけ）
ok = (df["estimate_jpy"] > 0) & (df["actual_jpy"] > 0)
r = np.log(df.loc[ok, "actual_jpy"] / df.loc[ok, "estimate_jpy"])
if len(r) >= 3:
    m, se = r.mean(), r.std(ddof=1) / np.sqrt(len(r))
    t = stats.t.ppf(0.975, len(r) - 1)
    print(f"実際 ÷ 見積もり：幾何平均 {np.exp(m):.2f} 倍"
          f"（95%区間 {np.exp(m - t*se):.2f}〜{np.exp(m + t*se):.2f} 倍、n={len(r)}）")
    print("  → 次からは、見積もりにこの区間の上側をかけた額を予算として確保する")
else:
    print(f"見積もりと実際が両方ある記録は {len(r)} 件。3件以上たまると比の区間を出します。")

In [ ]:
%%writefile cost_ledger.csv
date,service,purpose,gpu,hours,estimate_jpy,actual_jpy,storage_jpy,commit,notes
2026-11-28,colab-free,toy01 学習,T4,1.5,0,0,0,abc1234,無料枠。切断1回→再開OK

In [ ]:
%%writefile service_survey.csv
checked_date,service,plan_or_gpu,price,price_unit,storage_price,egress_price,prepaid,budget_alert,source_url,notes
2026-11-28,（例）サービス名,（例）GPU名,（調べた値）,円/時 または USD/時,（調べた値）,（調べた値）,yes/no,yes/no,https://...,為替レートも記録

In [ ]:
import glob, json, math, os, re, shlex, subprocess, sys
import numpy as np


def run(args):
    """ターミナルの「uv run python ○○.py …」の代わり。表示を出し、あとで確かめられるように文字列で返す"""
    p = subprocess.run([sys.executable] + shlex.split(args), capture_output=True, text=True)
    print(p.stdout, end="")
    if p.stderr.strip():
        print(p.stderr, file=sys.stderr)
    assert p.returncode == 0, "★実行に失敗：" + args
    return p.stdout


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, text, i=1):
    """表示（text）から、pattern の (…) の数を取り出す"""
    m = re.search(pattern, text.replace(",", ""))
    assert m, "表示に見つからない：" + pattern
    return float(m.group(i))


import torch
DEV = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("bench_step.py が選ぶ device:", DEV)

## 1. 1ステップの時間を測る（手順 3・4）

`bench_step.py` は、学習の1ステップ（順伝播・逆伝播・Adam の更新）を `--warmup` 回まわしてから `--steps` 回測り、平均と 95% 区間（$\bar t \pm 1.96\,s/\sqrt{n}$）、前半と後半の平均、ピークメモリを JSON に書きます。GPU は非同期に動くので、測る前後で `synchronize()` しています。

確かめること（手順 3）：「JSON に step_mean_s と95%区間が記録される。区間の幅が平均の数%以内に収まっている（広ければ --steps を増やす）」。CPU では他のプログラムの影響で区間が広くなることがあるので、ここでは表示だけにし、`ok()` は区間の計算が式どおりかを確かめます。

In [ ]:
# 教科書の設定：--model unet --batch 8 --size 256（--steps 50 が既定）。CPU では 1 ステップ数秒かかるので、ここでは一辺 128 にする
U_SIZE = 64 if DEV == "cpu" else 128           # CPU ではさらに小さく（一辺 64）
out = run(f"bench_step.py --model unet --batch 8 --size {U_SIZE} --steps 30 --warmup 5 --out bench_nb_unet.json")
b = json.load(open("bench_nb_unet.json", encoding="utf-8"))
for k in ("step_mean_s", "step_sd_s", "step_ci95_s", "step_mean_first_half_s", "step_mean_second_half_s", "samples_per_s", "peak_mem_mb", "params"):
    assert k in b, k
ok(True, "JSON に step_mean_s・step_ci95_s・前半／後半・samples_per_s・peak_mem_mb がある")
half = 1.96 * b["step_sd_s"] / math.sqrt(b["steps"])
ok(abs((b["step_ci95_s"][1] - b["step_ci95_s"][0]) / 2 - half) < 1e-12, "区間の半分の幅 = 1.96 × s / √n（式どおり）")
print(f"区間の幅は平均の {2 * half / b['step_mean_s'] * 100:.1f} %（数 % 以内でなければ --steps を増やす）")
ok(abs(b["samples_per_s"] - 8 / b["step_mean_s"]) < 1e-9, "処理量 = バッチサイズ ÷ 1ステップの平均時間")
ok(1.9e6 < b["params"] < 2.0e6, f"TinyUNet（w=32）のパラメータ数 {b['params'] / 1e6:.2f} M（プログラムの表示 1.93M）")

In [ ]:
# 教科書の設定：--model audio --batch 128 --size 4096（CPU ではバッチを 32 に小さくする）
A_BATCH = 32 if DEV == "cpu" else 128
out = run(f"bench_step.py --model audio --batch {A_BATCH} --size 4096 --steps 30 --warmup 5 --out bench_nb_audio.json")
a = json.load(open("bench_nb_audio.json", encoding="utf-8"))
ok(a["model"] == "audio" and a["size"] == 4096 and a["steps"] == 30, "1次元CNN（AudioCNN）の結果も JSON に保存した")
print(f"前半 {a['step_mean_first_half_s'] * 1e3:.1f} ms ／ 後半 {a['step_mean_second_half_s'] * 1e3:.1f} ms"
      "（手順 4：MacBook Air はファンがないので、--steps 600 のように長く測ると後半が遅くなることがある）")

## 2. バッチサイズとピークメモリ（手順 5）

手順 5：「バッチサイズを2、4、8と変えて bench_step.py を実行し、peak_mem_mb を記録します（MPSの値は目安）」。傾きが1サンプルあたりのメモリ $m_\text{act}$ の目安になります（III-1-3-2 の $M \approx P(b_w+b_g+b_o) + B\,m_\text{act} + M_0$）。

- **CUDA**（Colab の GPU）では `max_memory_allocated()` を使うので、ほぼ直線になります。ここで直線を当てはめ、`ok()` で傾きが正であることを確かめます。
- **MPS** の値は、キャッシュを含むドライバのメモリなので、バッチサイズを変えてもほとんど変わらないことがあります（III-1-7 の表「MPSのメモリの値が直線的に増えない」）。
- **CPU** では `peak_mem_mb` は `null` になります（測る関数がない）。このときは、この節は表示だけです。

In [ ]:
# 教科書の設定：--model unet --size 256 で --batch 2・4・8（各 --steps 50）。ここでは一辺 128、--steps 30 --warmup 3
rows = []
if DEV == "cpu":
    print("CPU では peak_mem_mb を測れないので、この節はとばします（Colab の GPU か Mac の MPS で試してください）")
else:
    for bs in (2, 4, 8):
        run(f"bench_step.py --model unet --batch {bs} --size 128 --steps 30 --warmup 3 --out bench_nb_mem{bs}.json")
        j = json.load(open(f"bench_nb_mem{bs}.json", encoding="utf-8"))
        rows.append((bs, j["peak_mem_mb"]))
    print("バッチサイズとピークメモリ [MB]:", rows)
    B_, M_ = np.array(rows, dtype=float).T
    slope, icpt = np.polyfit(B_, M_, 1)
    print(f"直線の当てはめ：1サンプルあたり {slope:.1f} MB、固定分 {icpt:.0f} MB")
    if DEV == "cuda":
        ok(slope > 0, "CUDA ではバッチサイズに対してピークメモリが増える（直線の傾き > 0。傾き = m_act の目安）")
    else:
        print("MPS の値はキャッシュを含む目安なので、直線にならなくても異常ではありません（正確な値は CUDA で測る）")

## 3. 費用を区間つきで見積もる（手順 6、図III-1-1）

`cost_estimator.py` は、ステップ時間の平均の不確かさ（正規近似）と、Mac の結果から GPU を推すときの速度倍率の幅（`--speedup 5 20` を 90% の範囲とする対数正規分布）から費用の分布を作り、5%点・中央値・95%点と予算を超える確率を出します。

まず、図III-1-1 の**仮の値**で計算します。ベンチマークの JSON は、図と同じ値（Mac：1ステップ 1.5 秒・標準偏差 0.1 秒・50 回、GPU のパイロット実行：0.18 秒・0.012 秒・50 回）を書いた見本です。図の説明：「(a) 90%区間は 160〜535 円と3倍以上の幅があり、予算を超える確率は約 7%。(b) GPU で10分だけ測ると（0.18 秒）、区間は 330〜340 円に狭まる」。

In [ ]:
# 図III-1-1 の仮の値（bench_step.py の出力と同じ名前の項目だけを書いた見本）
json.dump({"device_name": "Apple MPS（仮の値）", "model": "unet", "batch": 8, "steps": 50, "step_mean_s": 1.5, "step_sd_s": 0.1},
          open("bench_fig_mac.json", "w", encoding="utf-8"), ensure_ascii=False)
json.dump({"device_name": "GPU のパイロット実行（仮の値）", "model": "unet", "batch": 8, "steps": 50, "step_mean_s": 0.18, "step_sd_s": 0.012},
          open("bench_fig_gpu.json", "w", encoding="utf-8"), ensure_ascii=False)
out_a = run("cost_estimator.py --bench bench_fig_mac.json --speedup 5 20 --steps 100000 --price-jpy-h 60 --budget-jpy 500")
print()
out_b = run("cost_estimator.py --bench bench_fig_gpu.json --steps 100000 --price-jpy-h 60 --budget-jpy 500")
lo_a, hi_a = num(r"費用\s+: 中央値 [\d]+ 円（90%区間 (\d+)〜", out_a), num(r"費用\s+: 中央値 [\d]+ 円（90%区間 \d+〜(\d+)", out_a)
p_a = num(r"超える確率 ≈ ([\d.]+) %", out_a)
lo_b, hi_b = num(r"費用\s+: 中央値 [\d]+ 円（90%区間 (\d+)〜", out_b), num(r"費用\s+: 中央値 [\d]+ 円（90%区間 \d+〜(\d+)", out_b)
ok(abs(lo_a - 160) <= 3 and abs(hi_a - 535) <= 5 and hi_a / lo_a > 3, f"(a) Mac の結果だけ：90%区間 {lo_a:.0f}〜{hi_a:.0f} 円（図 160〜535 円、3倍以上の幅）")
ok(abs(p_a - 7.0) < 0.5, f"(a) 予算 500 円を超える確率 {p_a} %（図 約 7 %）")
ok(abs(lo_b - 330) <= 2 and abs(hi_b - 340) <= 2, f"(b) GPU で測った：90%区間 {lo_b:.0f}〜{hi_b:.0f} 円（図 330〜340 円）")
ok(num(r"止め忘れ12時間の費用 = (\d+) 円", out_a) == 720, "止め忘れ12時間の費用 = 60 円/時 × 12 時間 = 720 円")
med_b = num(r"費用\s+: 中央値 (\d+) 円", out_b)
ok(abs(med_b - 60 * (0.18 * 100000 / 3600 + 35 / 60)) < 1, f"(b) の中央値 {med_b:.0f} 円 = 60 円/時 ×（0.18 秒 × 100,000 ÷ 3600 ＋ setup 20 分 ＋ idle 15 分）")

次に、1 節で自分の機械で測った `bench_nb_unet.json` を使って、同じように見積もります（Mac・CPU の結果なら `--speedup 5 20` の幅をつけます。Colab の GPU で測った結果なら、`--speedup` は要りません）。時間単価 60 円/時は仮の値です。

In [ ]:
opt = "" if DEV == "cuda" else "--speedup 5 20 "
out = run(f"cost_estimator.py --bench bench_nb_unet.json {opt}--steps 30000 --price-jpy-h 60 --budget-jpy 1000")
c5, c95 = num(r"90%区間 (\d+)〜", out.split("費用")[1]), num(r"90%区間 \d+〜(\d+)", out.split("費用")[1])
ok(c5 <= num(r"費用\s+: 中央値 (\d+)", out) <= c95, "費用の 5%点 ≤ 中央値 ≤ 95%点 と、予算を超える確率が表示された")

## 4. 買う場合と借りる場合（手順 7、図III-1-2）

本文 III-1-3-5：「$P_\text{PC} = 315{,}000$ 円、$R = 0$、$p_h = 60$ 円/時、$p_e = 11$ 円/時（すべて仮の値）なら、$H^\ast \approx 6{,}400$ 時間」「buy_vs_rent.py は … 1.2倍になると見込んで（--cloud-overhead 1.2）計算するので、同じ60円/時でも約5,150時間と表示されます（--cloud-overhead 1 にすると上の式と同じ約6,400時間）」。図III-1-2 の説明では 約 6,409 時間・約 5,151 時間です。

In [ ]:
out12 = run("buy_vs_rent.py")
print()
out10 = run("buy_vs_rent.py --cloud-overhead 1")
pe = num(r"電気代 ≈ ([\d.]+) 円/時", out12)
h12, h10 = num(r"\s60 円/時 \|\s+(\d+)", out12), num(r"\s60 円/時 \|\s+(\d+)", out10)
ok(abs(pe - 10.85) < 0.06, f"自宅 PC の電気代 350 W × 31 円/kWh = {pe} 円/時（本文の約 11 円/時）")
ok(h12 == 5151 and h10 == 6409, f"60 円/時の損益分岐：×1.2 で {h12:.0f} 時間、×1 で {h10:.0f} 時間（図の 5,151・6,409 時間）")
yrs = num(r"\s60 円/時 \|\s+\d+\s+\|\s+[\d.]+ 年 \|\s+([\d.]+) 年", out10)
ok(abs(yrs - 6409 / (5 * 52)) < 0.1 and abs(yrs - 25) < 0.5, f"週5時間なら {yrs} 年（本文の約25年）")

## 5. 費用の台帳（手順 8・9）

`cost_ledger.csv` は教科書の見本の1行（Colab の無料枠）だけなので、`ledger_summary.py` は「見積もりと実際が両方ある記録は 0 件」と表示します。確かめること（手順 9）：「予算に対する使用率が表示される。見積もりと実際の比の区間は、記録が3件以上たまると表示される」。

そのあと、ノートブック版で**架空の3行**（有料の作業。見積もりと実際の費用は作り話の値）を足した `cost_ledger_demo.csv` で、比の区間が出ることを確かめます。$\ln r$（$r$ = 実際 ÷ 見積もり）の平均と $t$ 区間を、指数で元に戻した「幾何平均」とその区間です。

In [ ]:
out = run("ledger_summary.py cost_ledger.csv 3000")
ok("予算の 0%" in out and "両方ある記録は 0 件" in out, "見本の1行だけ：予算の使用率 0 %、比の区間はまだ出ない")
import pandas as pd
led = pd.read_csv("cost_ledger.csv")
demo = pd.concat([led, pd.DataFrame([   # 架空の記録（値は作り話。自分の台帳では消す）
    dict(date="2026-12-05", service="（架空）サービスA", purpose="unet 本番1", gpu="（架空）", hours=3.2, estimate_jpy=250, actual_jpy=310, storage_jpy=20, commit="0000000", notes="ノートブックの架空の行"),
    dict(date="2026-12-12", service="（架空）サービスA", purpose="unet 本番2", gpu="（架空）", hours=2.5, estimate_jpy=200, actual_jpy=260, storage_jpy=10, commit="0000000", notes="ノートブックの架空の行"),
    dict(date="2027-01-09", service="（架空）サービスB", purpose="ilrma 比較", gpu="（架空）", hours=1.0, estimate_jpy=120, actual_jpy=130, storage_jpy=0, commit="0000000", notes="ノートブックの架空の行"),
])], ignore_index=True)
demo.to_csv("cost_ledger_demo.csv", index=False)
out = run("ledger_summary.py cost_ledger_demo.csv 3000")
r = np.log(np.array([310 / 250, 260 / 200, 130 / 120]))
from scipy import stats
gm = math.exp(r.mean())
hi = math.exp(r.mean() + stats.t.ppf(0.975, 2) * r.std(ddof=1) / math.sqrt(3))
ok(abs(num(r"幾何平均 ([\d.]+) 倍", out) - round(gm, 2)) < 0.006 and abs(num(r"〜([\d.]+) 倍、n=3", out) - round(hi, 2)) < 0.006,
   f"記録が3件たまると、実際 ÷ 見積もりの幾何平均 {gm:.2f} 倍と 95% 区間（上側 {hi:.2f} 倍）が出る（式で計算し直しても同じ）")
ok(re.search(r"2026-12: 600 円（予算の 20%）", out) is not None, "月ごとの合計（実際の費用 ＋ ストレージ代：2026-12 は 310＋20＋260＋10 = 600 円）と予算の使用率 20 %")

調査シート `service_survey.csv` は、書き方の見本の1行だけです。値は、各サービスの公式サイトで調べた日付・出典の URL と一緒に自分で書きます（手順 8）。

In [ ]:
sv = pd.read_csv("service_survey.csv")
print(sv.T)
ok({"checked_date", "source_url", "price", "price_unit"} <= set(sv.columns), "調査シートに「調べた日付」「出典のURL」「価格」「単位」の列がある")

## 6. 原理の式の数値を確かめる（III-1-3）

- VRAM の見積もり $M \approx P(b_w+b_g+b_o) + B\,m_\text{act} + M_0$：float32 と Adam なら1パラメータあたり $4+4+8 = 16$ バイト。1 節の TinyUNet のパラメータ数で、重み・勾配・Adam の状態だけのメモリを出します（活性化 $B\,m_\text{act}$ のほうがずっと大きくなりがち）。
- 学習時間 $T \approx E\lceil N/B \rceil \bar t$。
- 50 ステップ測ったときの平均の 95% 区間 $\bar t \pm 1.96\,s/\sqrt{50}$。

In [ ]:
P = b["params"]
state_mb = P * 16 / 2**20
print(f"TinyUNet：P = {P:,}、重み＋勾配＋Adam の状態 = {P} × 16 バイト = {state_mb:.1f} MB")
ok(abs(state_mb - P * (4 + 4 + 8) / 2**20) < 1e-9 and state_mb < 100, "float32 と Adam で 1 パラメータ 16 バイト。この大きさのモデルなら数十 MB で、活性化の項のほうが大きい")
E, N, Bt, tbar = 10, 20000, a["batch"], a["step_mean_s"]
T = E * math.ceil(N / Bt) * tbar
print(f"1次元CNN：E = {E}、N = {N}、B = {Bt}、t̄ = {tbar * 1e3:.1f} ms → T ≈ {E} × {math.ceil(N / Bt)} × t̄ = {T / 60:.1f} 分")
print(f"（式の確認）バッチ 128 なら ⌈20000/128⌉ = {math.ceil(20000 / 128)} ステップ/エポック")
s = 0.1
ok(abs(1.96 * s / math.sqrt(50) - 0.0277) < 0.0001, "s = 0.1 秒、50 ステップなら、平均の区間は ±1.96 × 0.1/√50 ≈ ±0.028 秒")

## まとめ

- `bench_step.py` で1ステップの時間を区間つきで測り、`cost_estimator.py` で費用を区間つきで見積もりました。図III-1-1・図III-1-2 の仮の値の数値（160〜535 円・約7%、330〜340 円、5,151・6,409 時間）は、教科書のプログラムで再現できます。
- 実際の時間単価・ストレージ代・転送料は、各サービスの公式サイトで調べ、調べた日付と出典を `service_survey.csv` に書いてください（料金は変わります）。
- このノートブックに入れていないもの：GitHub のリポジトリ作り、予算と支払い方法を保護者と決めること、クラウドの GPU を借りること（III-3 以降）。